# data_001  –  KIER 에너지 데이터 전처리 파이프라인
`Data_01` hList → `Data_02` Wide ACCU → `Data_03` INST 계산 →
`Data_04` 보간 → `Data_05` IQR → `Data_06` ACCU 복원 → `Data_07` 리샘플 → `Data_99` 통합

## 01. Init

In [ ]:
#region Import_Basic
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
pd.options.display.float_format = '{:.10f}'.format

import random
import datetime as dt
from datetime import datetime, date, timedelta

import matplotlib.pyplot as plt, seaborn as sns
plt.rcParams['figure.figsize'] = [10, 8]

from tqdm.notebook import tqdm
#endregion

In [ ]:
from core import (data_datetime as com_date, data_preprocessing as com_Prep,
                  data_pipeline as com_Pipeline,
                  provider_kier_m02 as com_KIER_M02)

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

## 02. hList 생성

##### Case 01. KIER_hList_Common이 존재하지 않을 때  
공통 hList가 추출되지 않은 경우

In [ ]:
df_kier_hList_com = pd.DataFrame()
for i in range(0, 6):
    ## ▶ Define Variable and Directories
    ## Dict_Domain
    ## dict_domain = {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 5:"GAS"} ## GAS는 사용하지 않음.
    int_domain = i
    ## Domain, ACCU/INST Column
    str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
    ## Directory Root
    str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
    ## Interval, Target File
    str_interval, str_fileRaw, str_file_hList, str_file = com_KIER_M02.create_file_str(str_domain)

    print("■ " + str_domain)
    df_kier_hList_tmp = pd.read_csv(str_dir_raw + str_fileRaw, index_col = 0, error_bad_lines = False).drop_duplicates(subset = ['HOUSE_ID_DONG', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO']).reset_index()[['HOUSE_ID_DONG', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO']]

    ## "KeyError : 'HOUSE_ID_DONG'" 방지 : 최초 df_kier_hList_com는 빈 DataFrame이므로
    if i == 0 : df_kier_hList_com = df_kier_hList_tmp
    else : df_kier_hList_com = pd.merge(df_kier_hList_com, df_kier_hList_tmp, how = 'inner', on = ['HOUSE_ID_DONG', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO'])

print(df_kier_hList_com.shape, ' /// ', df_kier_hList_com.columns)
str_file_hList = str('KIER_hList_Common.csv')
df_kier_hList_com.to_csv(str_dir_raw + str_file_hList)

##### Case 02. KIER_hList_Common이 존재할 때  

In [ ]:
str_file_hList = str('KIER_hList_Common.csv')
df_kier_hList_com = pd.read_csv(str_dir_raw + str_file_hList, index_col = 0)[['HOUSE_ID_DONG', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO']]
df_kier_hList_com

In [ ]:
## ▶ Create Feature
## Target 변수('HOUSE_ID') 생성을 위한 호실 변수('HOUSE_ID_HO_INT') Column 생성
df_kier_hList_com['HOUSE_ID'], df_kier_hList_com['HOUSE_ID_HO_INT'] = 0, 0
## List : Building
list_Bld = df_kier_hList_com[['HOUSE_ID_DONG']]['HOUSE_ID_DONG'].drop_duplicates()
for bld in list_Bld:
    ## List : Floor
    list_F = df_kier_hList_com[df_kier_hList_com['HOUSE_ID_DONG'] == bld]['HOUSE_ID_HO_PRE'].drop_duplicates()
    
    for f in list_F:
        ## List : House
        list_H = df_kier_hList_com[(df_kier_hList_com['HOUSE_ID_DONG'] == bld)
                                   & (df_kier_hList_com['HOUSE_ID_HO_PRE'] == f)]['HOUSE_ID_HO'].drop_duplicates()

        ## 'HOUSE_ID_HO'는 해시코드 형태로 나타나므로  
        ## 숫자를 부여하여 호실로서 임의재정의 및 hList 내에 저장
        int_house_num = 1
        for h in list_H:
            idx_h = df_kier_hList_com[(df_kier_hList_com['HOUSE_ID_DONG'] == bld)
                                      & (df_kier_hList_com['HOUSE_ID_HO_PRE'] == f)
                                      & (df_kier_hList_com['HOUSE_ID_HO'] == h)].index
            df_kier_hList_com['HOUSE_ID_HO_INT'].loc[idx_h] = int_house_num

            int_house_num = int_house_num + 1

## 'HOUSE_ID' 변수 생성
df_kier_hList_com['HOUSE_ID'] = df_kier_hList_com.apply(lambda row : '-'.join(map(str, [row.HOUSE_ID_DONG, row.HOUSE_ID_HO_PRE, row.HOUSE_ID_HO_INT])), axis = 1)

## 기존 hList에 HOUSE_ID가 취합된 DataFrame을 덮어 씌움
df_kier_hList_com.to_csv(str_dir_raw + str_file_hList)
df_kier_hList_com

#### Case 01. 각 Domain에 대한 실행

In [ ]:
## ▶ Define Variable and Directories
## Dict_Domain
## dict_domain = {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 5:"GAS"} ## GAS는 사용하지 않음.
int_domain = 0
## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
## Interval, Target File
str_interval, str_fileRaw, str_file_hList, str_file = com_KIER_M02.create_file_str(str_domain)

In [ ]:
## ▶ Load Raw Data
## Gas의 경우는 ParserError: Error tokenizing data 발생하므로 아래와 같이 Load
## on_bad_lines='warn' 으로 바꾸면 어느 행이 제외됐는지 경고 출력 가능
str_file_hList = str('KIER_hList_Common.csv')
df_kier_hList_com = pd.read_csv(str_dir_raw + str_file_hList, index_col = 0)
df_kier_raw = pd.read_csv(str_dir_raw + str_fileRaw, index_col= 0, on_bad_lines='skip').reset_index()

## df_kier_raw와 hList를 Merge 및 유효 Column 정리
## .dropna() 필요 : Gas의 경우, 세대 정보가 유효하지 않은 Data가 1,800건 이상 존재
df_kier_raw = pd.merge(df_kier_raw, df_kier_hList_com, how = 'left', on = ['HOUSE_ID_DONG', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO'])[['METER_DATE', 'HOUSE_ID', str_col_accu]].dropna()

print(df_kier_raw.shape, ' /// ', df_kier_raw.columns)
df_kier_raw

In [ ]:
## Export df_kier_raw
str_file = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv')
df_kier_raw.to_csv(str_dir_raw + str_file)

#### Case 02. 모든 Domain에 대한 실행

In [ ]:
for i in range(0, 6):
    ## ▶ Define Variable and Directories
    ## Dict_Domain
    dict_domain = {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 5:"GAS"} ## GAS는 사용하지 않음.
    int_domain = i
    ## Domain, ACCU/INST Column
    str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
    ## Directory Root
    str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
    ## Interval, Target File
    str_interval, str_fileRaw, str_file_hList, str_file = com_KIER_M02.create_file_str(str_domain)
    
    print("■ " + str_domain)
    ## ▶ Load Raw Data
    ## Gas의 경우는 ParserError: Error tokenizing data 발생하므로 아래와 같이 Load
    ## 단, warn_bad_lines는 True로 그대로 두어 어느 행이 사라졌는지 확인해야만 함
    str_file_hList = str('KIER_hList_Common.csv')
    df_kier_hList_com = pd.read_csv(str_dir_raw + str_file_hList, index_col = 0)
    df_kier_raw = pd.read_csv(str_dir_raw + str_fileRaw, index_col= 0, error_bad_lines = False).reset_index()

    ## df_kier_raw와 hList를 Merge 및 유효 Column 정리
    ## .dropna() 필요 : Gas의 경우, 세대 정보가 유효하지 않은 Data가 1,800건 이상 존재
    df_kier_raw = pd.merge(df_kier_raw, df_kier_hList_com, how = 'left', on = ['HOUSE_ID_DONG', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO'])[['METER_DATE', 'HOUSE_ID', str_col_accu]].dropna()

    ## Export df_kier_raw
    str_file = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv')
    df_kier_raw.to_csv(str_dir_raw + str_file)

    print(df_kier_raw.shape, ' /// ', df_kier_raw.columns)
    print(df_kier_raw.isna().sum())
df_kier_raw

In [ ]:
df_kier_raw.columns

### [BU] Old Code 저장

In [ ]:
# ## HOT_HEAT와 HOT_FLOW가 한 시트에 존재하는 경우
# str_fileRaw = 'KIER_RAW_HOT_2024-03-13.csv'
# df_hot_tmp = pd.read_csv(str_dir_raw + str_fileRaw, index_col = 0).reset_index()
# df_hot_tmp[['METER_DATE', 'HOUSE_ID_DONG', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO', 'HOT_ACCU_HEAT']].to_csv(str_dir_raw + 'KIER_RAW_HOT_HEAT_2024-03-13.csv')
# df_hot_tmp[['METER_DATE', 'HOUSE_ID_DONG', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO', 'HOT_ACCU_FLOW']].to_csv(str_dir_raw + 'KIER_RAW_HOT_FLOW_2024-03-13.csv')

In [ ]:
## [2024-03-13][미사용] HOT Flow와 Heat를 Raw 단계에서 분리해서 사용
## [필요시] HOT_Cleansed.csv를 분리하자!
# str_fileCleansed = str('KIER_RAW_HOT_Cleansed.csv')
# df_raw = pd.read_csv(str_dir_raw + str_fileCleansed
#                      , index_col = 0)

# str_file_hot_heat = str('KIER_RAW_HOT_HEAT_Cleansed.csv')
# df_hot_heat = df_raw[['METER_DATE', 'HOUSE_ID_DONG', 'HOUSE_ID_HO_INT', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO', 'HOT_ACCU_HEAT']]
# df_hot_heat = df_hot_heat.rename(columns = {'HOT_ACCU_HEAT' : 'HOT_HEAT_ACCU'})
# df_hot_heat.to_csv(str_dir_raw + str_file_hot_heat)

# str_file_hot_flow = str('KIER_RAW_HOT_FLOW_Cleansed.csv')
# df_hot_flow = df_raw[['METER_DATE', 'HOUSE_ID_DONG', 'HOUSE_ID_HO_INT', 'HOUSE_ID_HO_PRE', 'HOUSE_ID_HO', 'HOT_ACCU_FLOW']]
# df_hot_heat = df_hot_heat.rename(columns = {'HOT_ACCU_HEAT' : 'HOT_FLOW_ACCU'})
# df_hot_heat.to_csv(str_dir_raw + str_file_hot_flow)

## 03. Raw → Wide Format (ACCU)

In [ ]:
## Dict_Domain
int_domain = 0

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

## File
str_fileRaw, str_fileRaw_hList = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv'), str('KIER_hList_Common.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
## Load Dataset (hList)
## "KIER_01-01_Data_hList.ipynb"로부터 만들어진 Bld/F/H List
df_kier_hList = pd.read_csv(str_dir_raw + str_fileRaw_hList, index_col = 0)
# print(df_kier_hList.shape, ' /// ', df_kier_hList.columns)
# df_kier_hList

##### Case 01. "METER_DATE" Cleansing이 완료되지 않은 경우

In [ ]:
## ▶ Load Dataset (Energy Usage)
df_raw = pd.read_csv(str_dir_raw + str_fileRaw, index_col = 0)
print(df_raw.shape, ' /// ', df_raw.columns)
df_raw

In [ ]:
## 1) Date Column에 대한 유효성 검사 및 이상 Data에 대한 소거
list_errValues = com_date.list_invalid_dates(df_raw, 'METER_DATE')
print(len(list_errValues))

## 유효하지 않은 행 일괄 제거
df_raw = df_raw.drop(index=list_errValues)

print(df_raw.shape, ' /// ', df_raw.columns)
df_raw

In [ ]:
## Cleansed Data를 저장
str_file = str('KIER_RAW_' + str_domain + '_Cleansed.csv')
df_raw.to_csv(str_dir_raw + str_file)
df_raw

##### Case 02. "METER_DATE" Cleansing이 완료된 경우

In [ ]:
str_fileCleansed = str('KIER_RAW_' + str_domain + '_Cleansed.csv')
df_raw = pd.read_csv(str_dir_raw + str_fileCleansed, index_col = 0)
print(df_raw.shape, ' /// ', df_raw.columns)
df_raw

In [ ]:
## ▶ Raw(long-format) → Wide-format 단일 pivot
list_HID = df_kier_hList['HOUSE_ID'].drop_duplicates()

df_wide = com_Pipeline.raw_to_wide(df_raw, str_col_accu, list_HID)

print(f"shape     : {df_wide.shape}")
print(f"세대 수   : {len(df_wide.columns) - 1}")
print(f"기간      : {df_wide['METER_DATE'].min()} ~ {df_wide['METER_DATE'].max()}")
df_wide.head()

In [ ]:
## ▶ 결과 저장
str_file_wide = 'KIER_' + str_domain + '_ACCU_10MIN.csv'
df_wide.to_csv(str_dirName_h + str_file_wide)
# Parquet 저장 (선택 – 하위 호환이 필요 없을 때 권장)
# df_wide.to_parquet(str_dirName_h + str_file_wide.replace('.csv','.parquet'), index=False)

print(f"저장 완료  : {str_dirName_h + str_file_wide}")
print(f"rows = {len(df_wide):,}  |  cols = {len(df_wide.columns):,}")

In [ ]:
## 결측치 현황
house_cols = [c for c in df_wide.columns if c != 'METER_DATE']
total   = len(df_wide) * len(house_cols)
nan_cnt = df_wide[house_cols].isna().sum().sum()
print(f"전체 데이터 수 : {total:,}")
print(f"결측치 합계    : {nan_cnt:,}")
print(f"결측치 비율    : {nan_cnt / total:.4%}")
df_wide[house_cols].isna().sum().sort_values(ascending=False).head(10)

## 04. INST 계산

In [ ]:
## Dict_Domain
int_domain = 0

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

## File
str_fileRaw, str_fileRaw_hList = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv'), str('KIER_hList_Common.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
## "KIER_01-01_Data_hList.ipynb"로부터 만들어진 Bld/F/H List
df_kier_hList = pd.read_csv(str_dir_raw + str_fileRaw_hList, index_col = 0)
# print(df_kier_hList.shape, ' /// ', df_kier_hList.columns)
# df_kier_hList

In [ ]:
## Load Intergrated ACCU Usage
str_file_accu = 'KIER_' + str_domain + '_ACCU_10MIN.csv'
df_ACCU_Intergrated = pd.read_csv(str_dirName_h + str_file_accu, index_col = 0)
df_ACCU_Intergrated

In [ ]:
## 모든 호실의 사용량을 변수화하여 한 데이터셋에 Combine  (vectorized)
## 기존: 348 세대 × 96K 행 반복 (~33M) → diff(1).shift(-1) 단일 연산으로 대체
list_HID = df_kier_hList['HOUSE_ID'].drop_duplicates()
accu_cols = [f'{str_col_accu}_{h}' for h in list_HID]
inst_cols  = [f'{str_col_inst}_{h}' for h in list_HID]

# INST[i] = ACCU[i+1] - ACCU[i]  ←→  diff(1).shift(-1)  (마지막 행은 NaN)
df_inst_tmp = df_ACCU_Intergrated[accu_cols].diff(periods=1).shift(-1)
df_inst_tmp.columns = inst_cols

df_kier_h_Combined = pd.concat(
    [df_ACCU_Intergrated[['YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE']], df_inst_tmp],
    axis=1
)

str_file = 'KIER_' + str_domain + '_INST_10MIN.csv'
df_kier_h_Combined.to_csv(str_dirName_h + str_file)
print(df_kier_h_Combined.shape, ' /// ', df_kier_h_Combined.columns)
df_kier_h_Combined

## 음수값 제거

In [ ]:
## Re-Load Dataset
str_file = 'KIER_' + str_domain + '_INST_10MIN.csv'       
df_kier_h_Combined = pd.read_csv(str_dirName_h + str_file, index_col = 0)

In [ ]:
list_col_tar = list(df_kier_h_Combined.columns)[5:]
list_col_tar

In [ ]:
## 음수값 제거  (vectorized)
## 기존: 348 컬럼 × 96K 행 이중 루프 → boolean 마스킹으로 대체
cnt_minus = (df_kier_h_Combined[list_col_tar] < 0).sum().sum()
df_kier_h_Combined[list_col_tar] = df_kier_h_Combined[list_col_tar].where(
    df_kier_h_Combined[list_col_tar] >= 0, other=np.nan
)
print(cnt_minus)

In [ ]:
## Export Datasets
str_file = 'KIER_' + str_domain + '_INST_10MIN.csv'       
df_kier_h_Combined.to_csv(str_dirName_h + str_file)

#### 2024-07-30 기준 Domain별 결측치 수 
- ELEC : 3799
- HEAT : 2205
- WATER : 7
- HOT_FLOW : 60
- HOT_HEAT : 36
- GAS : 2662

In [ ]:
# ## [미사용] Interpolation 후 수행해야할 연산
# list_col = df_kier_h_Combined.columns[5:]
# print(list_col)
# df_kier_Calc = df_kier_h_Combined[list_col]

# ## 사용량 평균 및 합계 Column 추가
# df_kier_h_Combined['MEAN_OF_INST'], df_kier_h_Combined['SUM_OF_INST'] = df_kier_Calc.mean(axis = 1), df_kier_Calc.sum(axis = 1)

# str_file = 'KIER_' + str_domain + '_INST_10MIN.csv'       
# df_kier_h_Combined.to_csv(str_dirName_h + str_file)
# print(df_kier_h_Combined.isnull().sum())
# df_kier_h_Combined

## 05. 결측치 보간

#### Module import

In [ ]:
## Dict_Domain
int_domain = 5

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

## File
str_fileRaw, str_fileRaw_hList = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv'), str('KIER_hList_Common.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
## "KIER_01-01_Data_hList.ipynb"로부터 만들어진 Bld/F/H List
df_kier_hList = pd.read_csv(str_dir_raw + str_fileRaw_hList, index_col = 0)
# print(df_kier_hList.shape, ' /// ', df_kier_hList.columns)
# df_kier_hList

In [ ]:
## Load Integrated INST Usage  (Parquet 우선 → 없으면 CSV 폴백)
import pathlib as _pl

str_file_pq  = 'KIER_' + str_domain + '_INST_10MIN.parquet'
str_file_csv = 'KIER_' + str_domain + '_INST_10MIN.csv'

_path = _pl.Path(str_dirName_h + str_file_pq)
if _path.exists():
    df_INST_Intergrated = pd.read_parquet(_path)
    print(f'[parquet] {str_file_pq}')
else:
    df_INST_Intergrated = pd.read_csv(str_dirName_h + str_file_csv, index_col=0)
    print(f'[csv] {str_file_csv}')

df_INST_Intergrated

In [ ]:
## 세대 컬럼 목록 추출 (METER_DATE / YEAR~MINUTE / 통계 컬럼 제외)
_META_COLS = {'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE',
              'MEAN_OF_INST', 'SUM_OF_INST'}
house_cols = [c for c in df_INST_Intergrated.columns if c not in _META_COLS]

## 사용량 평균·합계 컬럼 추가
df_INST_Intergrated['MEAN_OF_INST'] = df_INST_Intergrated[house_cols].mean(axis=1)
df_INST_Intergrated['SUM_OF_INST']  = df_INST_Intergrated[house_cols].sum(axis=1)

print(f'세대 수     : {len(house_cols)}')
print(f'결측치 합계 : {df_INST_Intergrated[house_cols].isna().sum().sum():,}')
df_INST_Intergrated

In [ ]:
## 세대별 결측치 현황
nan_per_house = df_INST_Intergrated[house_cols].isna().sum()
print(nan_per_house[nan_per_house > 0].sort_values(ascending=False))

In [ ]:
## 결측치 전체 통계
total   = len(df_INST_Intergrated) * len(house_cols)
nan_cnt = df_INST_Intergrated[house_cols].isna().sum().sum()
print(f'Total Data  : {total:,}')
print(f'Sum of Nan  : {nan_cnt:,}')
print(f'Rate of Nan : {nan_cnt / total:.4%}')

### 순시사용량 결측치 보간 (벡터화 3단계)
- **Stage 1**: 각 시각의 전체 세대 평균으로 NaN 대치
- **Stage 2**: all-NaN 시각(평균도 NaN)은 평균 선형보간 후 재대치
- **Stage 3**: 잔여 NaN 세대별 선형보간 (양방향)
- 기존 `for i in range(len(df))` row-loop(~96K 회) 대비 완전 벡터화

In [ ]:
## ▶ 결측치 보간 (벡터화 3단계)
df_stage1, df_final = com_Pipeline.fill_missing_wide(df_INST_Intergrated, house_cols)

nan_s1  = df_stage1[house_cols].isna().sum().sum()
nan_fin = df_final[house_cols].isna().sum().sum()
print(f'Stage 1 후 잔여 결측치 : {nan_s1:,}')
print(f'Stage 3 후 잔여 결측치 : {nan_fin:,}')
df_final

In [ ]:
## ▶ 결과 저장
str_file = 'KIER_' + str_domain + '_INST_10MIN_1st_Mean.csv'
df_stage1.to_csv(str_dirName_h + str_file)
# df_stage1.to_parquet(str_dirName_h + str_file.replace('.csv','.parquet'), index=False)

str_file = 'KIER_' + str_domain + '_INST_10MIN_2st_Linear.csv'
df_final.to_csv(str_dirName_h + str_file)
# df_final.to_parquet(str_dirName_h + str_file.replace('.csv','.parquet'), index=False)

print('저장 완료')
df_final

In [ ]:
## 보간 전/후 비교
print('=== Stage 1 (평균 대치) ===')
print(f'결측치 : {df_stage1[house_cols].isna().sum().sum():,}')
print()
print('=== Stage 3 (최종) ===')
print(f'결측치 : {df_final[house_cols].isna().sum().sum():,}')

In [ ]:
## 데이터 확인 (임의 세대 샘플)
sample_col = house_cols[0]
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(df_INST_Intergrated[sample_col].values, linewidth=0.5, color='tomato')
axes[0].set_title(f'Before Interpolation  |  {sample_col}')
axes[1].plot(df_final[sample_col].values, linewidth=0.5, color='steelblue')
axes[1].set_title('After Interpolation')
plt.tight_layout()
plt.show()

## 06. 이상치 제거 (IQR)

#### Module import

In [ ]:
## Dict_Domain
int_domain = 0

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

## File
str_fileRaw, str_fileRaw_hList = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv'), str('KIER_hList_Common.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
## "KIER_01-01_Data_hList.ipynb"로부터 만들어진 Bld/F/H List
df_kier_hList = pd.read_csv(str_dir_raw + str_fileRaw_hList, index_col = 0)
# print(df_kier_hList.shape, ' /// ', df_kier_hList.columns)
# df_kier_hList

In [ ]:
## Load Intergrated INST Usage
str_file_inst = 'KIER_' + str_domain + '_INST_10MIN_2st_Linear.csv'
str_dirName_h
df_INST_Intergrated = pd.read_csv(str_dirName_h + str_file_inst, index_col = 0)
df_INST_Intergrated = com_date.create_col_datetime(df_INST_Intergrated, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels=['None'], axis = 1)

## 각 호실별 사용량
list_col_tar = list(df_INST_Intergrated.columns)[6:-2]

df_INST_Intergrated

In [ ]:
## Target Column 확인
list_col_tar

## 데이터 확인
plt.plot(df_INST_Intergrated[str_col_inst + '_561-1-1'])
plt.title('Instantaneous Electric Usage in certain household')

## IQR Code

In [ ]:
## 여기서 IQR방식으로 전처리
for tar_col in list_col_tar:
    print(tar_col)
    df_INST_Intergrated = com_Prep.del_outlier_usages(df_INST_Intergrated, tar_col, max_iter=1)
print(df_INST_Intergrated.shape, ' /// ', df_INST_Intergrated.columns)
df_INST_Intergrated

In [ ]:
## 데이터 확인
plt.plot(df_INST_Intergrated[str_col_inst + '_561-1-1'])
plt.title('Instantaneous Electric Usage in certain household')
# plt.xlim(0, 5000)

In [ ]:
## Export Dataframe
str_file = 'KIER_' + str_domain + '_INST_03_IQR.csv'
df_INST_Intergrated.to_csv(str_dirName_h + str_file)

## 처리 단계별 데이터 비교

In [ ]:
str_file = 'KIER_' + str_domain + '_INST_01_10min.csv'
df_tmp_01 = pd.read_csv(str_dirName_h + str_file, index_col = 0)
str_file = 'KIER_' + str_domain + '_INST_10MIN_2st_Linear.csv'
df_tmp_02 = pd.read_csv(str_dirName_h + str_file, index_col = 0)
str_file = 'KIER_' + str_domain + '_INST_03_IQR.csv'
df_tmp_03 = pd.read_csv(str_dirName_h + str_file, index_col = 0)

period_tmp = df_tmp_03['METER_DATE']

In [ ]:
## list_col_tar = list(df_INST_Intergrated.columns)[6:-2] ## 각 세대별 사용량

for tar_col in list_col_tar[0:5]:
    plt.figure(figsize=(20,4))

    ## Phase 2 vs Phase 1 (Interpolated vs Raw)
    plt.plot(df_tmp_02[tar_col], color='blue', label='Mean Interpolated')
    plt.plot(df_tmp_01[tar_col], color='red', label='Raw')
    # plt.plot(df_tmp_03[tar_col], color='green', label='IQR')

    ## 상세보기를 위한 범위 축소
    plt.xlim(0, 5000)
    plt.legend()
    plt.show()

In [ ]:
for tar_col in list_col_tar[0:5]:
    plt.figure(figsize=(20,4))

    ## Phase 2 vs Phase 3 (Interpolated vs IQR)
    # plt.plot(df_tmp_01[tar_col], color='red', label='Raw')
    plt.plot(df_tmp_02[tar_col], color='blue', label='Mean Interpolated')
    plt.plot(df_tmp_03[tar_col], color='green', label='IQR')
    
    ## 상세보기를 위한 범위 축소
    plt.xlim(0, 5000)
    plt.legend()
    plt.show()

In [ ]:
for tar_col in list_col_tar[0:5]:
    plt.figure(figsize=(20,4))

    ## Phase 1 vs Phase 3 (Raw vs IQR)
    # plt.plot(df_tmp_02[tar_col], color='blue', label='Mean Interpolated')
    plt.plot(df_tmp_01[tar_col], color='red', label='Raw')
    plt.plot(df_tmp_03[tar_col], color='green', label='IQR')

    ## 상세보기를 위한 범위 축소
    plt.xlim(0, 5000)
    plt.ylim(0, 0.5)
    plt.legend()
    plt.show()

## Column 갱신 : SUM_OF_INST / MEAN_OF_INST

In [ ]:
df_kier_Calc = df_INST_Intergrated[list_col_tar]
df_INST_Intergrated['MEAN_OF_INST'], df_INST_Intergrated['SUM_OF_INST'] = df_kier_Calc.mean(axis = 1), df_kier_Calc.sum(axis = 1)
df_INST_Intergrated

In [ ]:
## Export Dataframe
str_file = 'KIER_' + str_domain + '_INST_03_IQR.csv'
df_INST_Intergrated.to_csv(str_dirName_h + str_file)

## 07. ACCU 복원

#### Module import

In [ ]:
## Dict_Domain
int_domain = 5

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

## File
str_fileRaw, str_fileRaw_hList = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv'), str('KIER_hList_Common.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
## "KIER_01-01_Data_hList.ipynb"로부터 만들어진 Bld/F/H List
df_kier_hList = pd.read_csv(str_dir_raw + str_fileRaw_hList, index_col = 0)
# print(df_kier_hList.shape, ' /// ', df_kier_hList.columns)
list(df_kier_hList['HOUSE_ID'])
# df_kier_hList['HOUSE_ID']

In [ ]:
## Load Intergrated ACCU Usage
str_file_accu = 'KIER_' + str_domain + '_ACCU_10MIN.csv'
## Load Intergrated INST Usage
str_file_inst = 'KIER_' + str_domain + '_INST_10MIN_2st_Linear.csv'

df_accu = pd.read_csv(str_dirName_h + str_file_accu, index_col = 0)
df_inst = pd.read_csv(str_dirName_h + str_file_inst, index_col = 0)

# df_INST_Intergrated = com_date.create_col_datetime(df_INST_Intergrated, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels=['None'], axis = 1)

## 각 호실별 사용량
list_col_tar_accu = list(df_accu.columns)[5:-2]
list_col_tar_inst = list(df_inst.columns)[5:-2]

print(df_accu.isna().sum().sum())
print(df_inst.isna().sum().sum())
df_accu
df_inst

In [ ]:
## "평균대치" 보간 (확인차 재보간)  (vectorized)
## 기존: 96K 행 루프 (ELEC ~30분) → apply 열 단위 fillna 로 대체
df_inst[list_col_tar_inst] = df_inst[list_col_tar_inst].apply(
    lambda col: col.fillna(df_inst['MEAN_OF_INST'])
)

In [ ]:
## "단순Linear -> 단순 0 대입" 보간 (확인차 재보간)
df_inst[list_col_tar_inst] = df_inst[list_col_tar_inst].interpolate(axis=0)  # vectorized
df_inst = df_inst.fillna(0)
df_inst

In [ ]:
df_accu

In [ ]:
## 각 호실별 최소 적산치를 첫 행의 에너지 사용량에 대입  (vectorized)
## 기존: 348 세대 루프 + math.isnan() → boolean 마스킹으로 대체
first_row      = df_accu.loc[df_accu.index[0], list_col_tar_accu]
first_nan_mask = first_row.isna()
df_accu.loc[df_accu.index[0], list_col_tar_accu] = first_row.where(
    ~first_nan_mask, df_accu[list_col_tar_accu].min()
)
df_accu

In [ ]:
## 각 호실별 보간된 순시사용량을 적산치에 적용  (vectorized)
## 기존: 348 세대 × 96K 행 중첩 루프 (~80~90분) → cumsum 단일 연산으로 대체
## ACCU[i] = ACCU[0] + INST[0] + INST[1] + ... + INST[i-1]
##         = ACCU[0] + cumsum(INST.shift(1).fillna(0))[i]
accu_init  = df_accu[list_col_tar_accu].iloc[[0]].values           # shape (1, N_houses)
inst_csum  = df_inst[list_col_tar_inst].shift(1).fillna(0).cumsum()
df_accu[list_col_tar_accu] = accu_init + inst_csum.values
df_accu

In [ ]:
df_accu = com_date.create_col_datetime(df_accu, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE')
df_accu

In [ ]:
df_accu.isna().sum().sum()

In [ ]:
str_file_accu = 'KIER_' + str_domain + '_ACCU_10MIN_Restored.csv'
df_accu.to_csv(str_dirName_h + str_file_accu)

## 08-1. 리샘플링 (ACCU)

 - CODE : Resample - KIER Energy Usage  
 - DESC  
    &ensp; 1) "M02-01_Data_06_Restore_Energy ACCU.ipynb"로부터 복원된 에너지 적산 사용량을 Resample  
    &ensp; 2) Resample된 데이터를 기반으로 기간별(10MIN, 30MIN, 1H, 12H, 24H, 1W, 2W, 1M) 순시 사용 데이터 산출 및 CSV 추출  

 - DATE  
    &ensp; 2024-08-05 Created  

In [ ]:
## Dict_Domain
int_domain = 5

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

## File
str_fileRaw, str_fileRaw_hList = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv'), str('KIER_hList_Common.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
## Combined Data
str_file = 'KIER_' + str_domain + '_ACCU_10MIN_Restored.csv'

df_kier_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)
# df_kier_raw = com_date.create_col_datetime(df_kier_raw, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels=['None'], axis = 1)
df_kier_raw['METER_DATE'] = pd.to_datetime(df_kier_raw['METER_DATE'])
df_kier_raw = df_kier_raw[['METER_DATE'] + list(df_kier_raw.columns[6:])]
df_kier_raw

In [ ]:
## 시간 간격 (10분 / 30분 / 1시간 / 1일 / 1주 / 1달)
list_col_interval = ['10T', '30T', '1H', '12H', '1D', '1W', '2W', '1M']
for str_int in list_col_interval:
    print(str_int)
    df_kier_res = df_kier_raw.resample(str_int, on = 'METER_DATE').last().reset_index()

    if str_int == '10T' : str_int = '10MIN'
    elif str_int == '30T' : str_int = '30MIN'

    str_file = 'KIER_' + str_domain + '_Resample_ACCU_' + str_int + '.csv'
    df_kier_res.to_csv(str_dirName_h + str_file)
df_kier_res

## 08-2. 리샘플링 (INST)

 - CODE : Resample - KIER Energy Usage  
 - DESC  
    &ensp; 1) "M02-01_Data_06_Restore_Energy ACCU.ipynb"로부터 복원된 에너지 적산 사용량을 Resample  
    &ensp; 2) Resample된 데이터를 기반으로 기간별(10MIN, 30MIN, 1H, 12H, 24H, 1W, 2W, 1M) 순시 사용 데이터 산출 및 CSV 추출  

 - DATE  
    &ensp; 2024-08-05 Created  

In [ ]:
## Dict_Domain
int_domain = 5

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

## File
str_fileRaw, str_fileRaw_hList = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv'), str('KIER_hList_Common.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
## "KIER_01-01_Data_hList.ipynb"로부터 만들어진 Bld/F/H List
df_kier_hList = pd.read_csv(str_dir_raw + str_fileRaw_hList, index_col = 0)
# print(df_kier_hList.shape, ' /// ', df_kier_hList.columns)
# list(df_kier_hList['HOUSE_ID'])
# df_kier_hList['HOUSE_ID']

In [ ]:
## 시간 간격 (10분 / 30분 / 1시간 / 1일 / 1주 / 1달) 
list_col_interval = ['10T', '30T', '1H', '12H', '1D', '1W', '2W', '1M']
# list_col_interval = ['10T', '1H', '12H', '1D', '1W', '2W', '1M']
# list_col_interval = ['1M']
for str_int in list_col_interval:
    print(str_int)
    if str_int == '10T' : 
        str_int = '10MIN'
        str_file_inst = 'KIER_' + str_domain + '_INST_10MIN_2st_Linear.csv'
        df_kier_inst = pd.read_csv(str_dirName_h + str_file_inst, index_col = 0)
        str_file_inst = 'KIER_' + str_domain + '_INST_' + str_int + '_Resampled.csv'
        df_kier_inst.to_csv(str_dirName_h + str_file_inst)
        continue
    elif str_int == '30T' : str_int = '30MIN'

    str_file_accu = 'KIER_' + str_domain + '_Resample_ACCU_' + str_int + '.csv'
    df_kier_accu = pd.read_csv(str_dirName_h + str_file_accu, index_col = 0)

    df_kier_inst = pd.DataFrame()
    for HID in list(df_kier_hList['HOUSE_ID']):
        print(HID)
        df_kier_inst['METER_DATE'] = df_kier_accu['METER_DATE']
        df_kier_inst[str_col_inst + '_' + HID] = 0
        
        for i in range(0, len(df_kier_accu) - 1) : 
                df_kier_inst[str_col_inst + '_' + HID].iloc[i] = df_kier_accu[str_col_accu + '_' + HID].iloc[i + 1] - df_kier_accu[str_col_accu + '_' + HID].iloc[i]

    str_file_inst = 'KIER_' + str_domain + '_INST_' + str_int + '_Resampled.csv'
    df_kier_inst[:-1].to_csv(str_dirName_h + str_file_inst)

df_kier_accu

In [ ]:
df_kier_inst[:-1]

## 09. 전체 통합

In [ ]:
## Dict_Domain
int_domain = 0

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

## File
str_fileRaw, str_fileRaw_hList = str('KIER_RAW_' + str_domain + '_H_ID_Adopted.csv'), str('KIER_hList_Common.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
## "KIER_01-01_Data_hList.ipynb"로부터 만들어진 Bld/F/H List
df_kier_hList = pd.read_csv(str_dir_raw + str_fileRaw_hList, index_col = 0)
# print(df_kier_hList.shape, ' /// ', df_kier_hList.columns)
# df_kier_hList

In [ ]:
## 최적의 Period 계산
dt_start, dt_end = 0, 0

## 각 호별 Resampled Data 생성 및 취합
# df_kier_h_Combined = df_dt.copy()
cnt_h, df_kier_h_Combined = pd.DataFrame(), 0

list_HID = df_kier_hList['HOUSE_ID'].drop_duplicates()
for house in list_HID:
    print(str(house) + " H")

    str_file_h = str('KIER_' + str_domain + '_' + str(house) + '_ACCU_01_Raw.csv')
    df_h_tmp = pd.read_csv(str_dirName_h + str_file_h, index_col = 0)

    if dt_start == 0: 
        dt_start = df_h_tmp['METER_DATE'].min()
        print(dt_start)
    else : 
        if dt_start < df_h_tmp['METER_DATE'].min(): 
            print("True Start")
            dt_start = df_h_tmp['METER_DATE'].min()
    
    if dt_end == 0: 
        dt_end = df_h_tmp['METER_DATE'].max()
        print(dt_end)
    else : 
        if dt_start > df_h_tmp['METER_DATE'].max(): 
            print("True End")
            dt_start = df_h_tmp['METER_DATE'].max()

print('Calculated Period')
print(dt_start, ' / ', dt_end)

In [ ]:
## 여기서부터 실행하려 할 경우, Period를 지정해줘야함
## 계산된 Period에 대한 Date Dataframe 생성
df_kier_h_Combined = com_date.create_df_dt(pd.DataFrame(), 'METER_DATE', dt_start, dt_end, '10min')[['YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE']]

## METER_DATE의 Minute이 10분 단위가 아닌 경우, 10분 단위로 변경
df_kier_h_Combined['MINUTE'] = (df_kier_h_Combined['MINUTE'] // 10) * 10  # vectorized
df_kier_h_Combined

In [ ]:
list_HID = df_kier_hList['HOUSE_ID'].drop_duplicates()
for house in list_HID:
    print(str(house) + " H")

    str_col_accu_h = str_col_accu + "_" + house
    str_file = 'KIER_' + str_domain + '_' + str(house) + '_ACCU_01_Raw.csv'
    df_h_tmp = pd.read_csv(str_dirName_h + str_file, index_col = 0).reset_index()[['YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE', str_col_accu_h]]

    df_kier_h_Combined = pd.merge(df_kier_h_Combined, df_h_tmp, how = 'left', on = ['YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE'])
    print(df_kier_h_Combined.shape, ' /// ', df_kier_h_Combined.columns)

str_file = 'KIER_' + str_domain + '_ACCU_10MIN.csv'
df_kier_h_Combined.to_csv(str_dirName_h + str_file)
print(df_kier_h_Combined.shape, ' /// ', df_kier_h_Combined.columns)
df_kier_h_Combined

In [ ]:
# ## [미사용] Interpolation 후 수행해야할 연산
# list_col = df_kier_h_Combined.columns[5:] ## 세대별 에너지 사용량 Columns
# df_kier_Calc = df_kier_h_Combined[list_col]
# print(list_col)

# ## df_kier_h_Combined는 차후 "04. 순시사용량" 과정에서 재활용되므로, 
# ## df_kier_extract를 별도 선언 및 추출
# df_kier_extract = df_kier_h_Combined

# ## 사용량 평균 및 합계 Column 추가
# df_kier_extract['MEAN_OF_ACCU'], df_kier_extract['SUM_OF_ACCU'] = df_kier_Calc.mean(axis = 1), df_kier_Calc.sum(axis = 1)

# str_file = 'KIER_' + str_domain + '_ACCU_10MIN.csv'
# df_kier_extract.to_csv(str_dirName_h + str_file)
# print(df_kier_extract.isnull().sum())
# df_kier_extract